In [ ]:
# ============================================
# STUDENT SCORE PREDICTION - COMPLETE PIPELINE
# ============================================

# ---------- 1. IMPORT REQUIRED LIBRARIES ----------

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Sklearn utilities
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Models
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor

# Advanced models
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

import warnings
warnings.filterwarnings("ignore")


# ---------- 2. LOAD DATASET ----------

# Load CSV file
df = pd.read_csv("data/stud.csv")

# Preview data
df.head()


# ---------- 3. SPLIT FEATURES & TARGET ----------

# X = input features
X = df.drop(columns=["math_score"], axis=1)

# y = target variable
y = df["math_score"]


# ---------- 4. CHECK CATEGORICAL VALUES ----------

print("Gender:", df["gender"].unique())
print("Race/Ethnicity:", df["race_ethnicity"].unique())
print("Parental Education:", df["parental_level_of_education"].unique())
print("Lunch:", df["lunch"].unique())
print("Test Preparation:", df["test_preparation_course"].unique())


# ---------- 5. PREPROCESSING PIPELINE ----------

# Identify numerical & categorical columns
num_features = X.select_dtypes(exclude="object").columns
cat_features = X.select_dtypes(include="object").columns

# Transformers
numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder()

# Column Transformer
preprocessor = ColumnTransformer(
    [
        ("OneHotEncoder", categorical_transformer, cat_features),
        ("StandardScaler", numeric_transformer, num_features)
    ]
)


# ---------- 6. APPLY TRANSFORMATION ----------

X = preprocessor.fit_transform(X)

# Check shape after encoding
X.shape


# ---------- 7. TRAIN-TEST SPLIT ----------

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train.shape, X_test.shape


# ---------- 8. MODEL EVALUATION FUNCTION ----------

def evaluate_model(true, predicted):
    """
    Returns MAE, RMSE, and R2 score
    """
    mae = mean_absolute_error(true, predicted)
    rmse = np.sqrt(mean_squared_error(true, predicted))
    r2 = r2_score(true, predicted)
    return mae, rmse, r2


# ---------- 9. TRAIN MULTIPLE MODELS ----------

models = {
    "Linear Regression": LinearRegression(),
    "Lasso": Lasso(),
    "Ridge": Ridge(),
    "KNN": KNeighborsRegressor(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest": RandomForestRegressor(),
    "XGBRegressor": XGBRegressor(),
    "CatBoost": CatBoostRegressor(verbose=False),
    "AdaBoost": AdaBoostRegressor()
}

model_names = []
r2_scores = []

for name, model in models.items():
    # Train model
    model.fit(X_train, y_train)

    # Predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    # Evaluate
    train_mae, train_rmse, train_r2 = evaluate_model(y_train, y_train_pred)
    test_mae, test_rmse, test_r2 = evaluate_model(y_test, y_test_pred)

    # Print results
    print(name)
    print(f"Train RMSE: {train_rmse:.4f}, R2: {train_r2:.4f}")
    print(f"Test  RMSE: {test_rmse:.4f}, R2: {test_r2:.4f}")
    print("=" * 40)

    model_names.append(name)
    r2_scores.append(test_r2)


# ---------- 10. MODEL COMPARISON ----------

results = pd.DataFrame({
    "Model": model_names,
    "R2 Score": r2_scores
}).sort_values(by="R2 Score", ascending=False)

results


# ---------- 11. FINAL MODEL (LINEAR REGRESSION) ----------

final_model = LinearRegression()
final_model.fit(X_train, y_train)

y_pred = final_model.predict(X_test)

final_r2 = r2_score(y_test, y_pred) * 100
print(f"Final Model Accuracy (R2): {final_r2:.2f}%")


# ---------- 12. PLOT ACTUAL vs PREDICTED ----------

plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.6)
plt.xlabel("Actual Math Score")
plt.ylabel("Predicted Math Score")
plt.title("Actual vs Predicted Scores")
plt.show()


FileNotFoundError: [Errno 2] No such file or directory: 'data/stud.csv'